# Tool Base

The `base.py` module defines the core interfaces, schemas, exceptions, annotations, and utility functions used by LangChain tools.

A tool is a Runnable component that receives structured or string input, performs an action, and returns an output. Tools can validate their inputs, emit callbacks, support synchronous and asynchronous execution, return artifacts, and handle execution or validation errors.

## Constants

1. `FILTERED_ARGS`: Stores internal function arguments that are excluded from generated tool schemas.
   * **Definition:**
     ```python
     FILTERED_ARGS = (
         "run_manager",
         "callbacks"
     )
     ```

2. `TOOL_MESSAGE_BLOCK_TYPES`: Stores the supported structured content-block types that may be returned as tool-message content.
   * **Definition:**
     ```python
     TOOL_MESSAGE_BLOCK_TYPES = (
         "text",
         "image_url",
         "image",
         "json",
         "search_result",
         "custom_tool_call_output",
         "document",
         "file"
     )
     ```

## Type Aliases

1. `ArgsSchema`: Represents a tool input schema as either a Pydantic model class or a JSON Schema dictionary.
   * **Definition:**
     ```python
     ArgsSchema = TypeBaseModel | dict[str, Any]
     ```

2. `MessageContentBlock`: Represents one tool-message content block as plain text or a structured dictionary.
   * **Definition:**
     ```python
     MessageContentBlock = str | dict[str, Any]
     ```

3. `ToolExceptionHandlerOutput`: Represents content returned by a custom tool-error handler.
   * **Definition:**
     ```python
     ToolExceptionHandlerOutput = (
         str
         | Sequence[MessageContentBlock]
     )
     ```

# SchemaAnnotationError

`SchemaAnnotationError` is raised when a tool's `args_schema` field is missing a valid type annotation or uses an incorrect annotation.

## Bases

- `TypeError`

# ToolException

`ToolException` represents an execution error that a tool may expose to an agent without necessarily stopping the entire agent loop.

The error is processed according to the tool's `handle_tool_error` setting.

## Bases

- `Exception`

## Functions

1. `create_schema_from_function`: Creates a Pydantic input schema from a Python function's signature.

   It can exclude internal parameters, parse Google-style docstrings, validate docstring argument names, and optionally exclude runtime-injected arguments.

   * **Syntax:**
     ```python
     create_schema_from_function(
         model_name: str, # Name assigned to the generated model
         func: Callable[..., Any], # Function whose signature is inspected
         *,
         filter_args: Sequence[str] | None = None, # Arguments excluded from the schema
         parse_docstring: bool = False, # Whether to parse argument descriptions
         error_on_invalid_docstring: bool = False, # Whether invalid docstrings raise errors
         include_injected: bool = True # Whether injected arguments remain in the schema
     ) -> TypeBaseModel
     ```

2. `get_all_basemodel_annotations`: Returns field annotations from a Pydantic model and its parent models.

   It resolves inherited generic type variables and supports both Pydantic v1 and v2 models.

   * **Syntax:**
     ```python
     get_all_basemodel_annotations(
         cls: TypeBaseModel | Any, # Pydantic model class to inspect
         *,
         default_to_bound: bool = True # Replace unresolved TypeVars with their bounds
     ) -> dict[str, type | TypeVar]
     ```

# BaseTool

`BaseTool` is the abstract base class for all LangChain tools.

It defines tool metadata, input validation, callback integration, synchronous and asynchronous execution, error handling, artifact handling, and Runnable compatibility.

## Bases

- `RunnableSerializable[str | dict[str, Any] | ToolCall, Any]`

## Attributes

1. `name`: Stores the unique name of the tool.
   * **Type:**
     ```python
     name: str
     ```

2. `description`: Explains what the tool does and when a model should use it.
   * **Type:**
     ```python
     description: str
     ```

3. `args_schema`: Stores the schema used to validate and parse tool inputs.

   It may be a Pydantic v1 model, a Pydantic v2 model, a JSON Schema dictionary, or `None`.

   * **Type:**
     ```python
     args_schema: Annotated[
         ArgsSchema | None,
         SkipValidation()
     ] = Field(
         default=None,
         description="The tool schema."
     )
     ```

4. `return_direct`: Controls whether an agent executor should stop its loop after the tool returns.
   * **Type:**
     ```python
     return_direct: bool = False
     ```

5. `verbose`: Controls whether tool execution progress is logged.
   * **Type:**
     ```python
     verbose: bool = False
     ```

6. `callbacks`: Stores callbacks invoked during tool execution.
   * **Type:**
     ```python
     callbacks: Callbacks = Field(
         default=None,
         exclude=True
     )
     ```

7. `tags`: Stores optional tags attached to each execution and passed to callback handlers.
   * **Type:**
     ```python
     tags: list[str] | None = None
     ```

8. `metadata`: Stores optional metadata attached to each execution and passed to callback handlers.
   * **Type:**
     ```python
     metadata: dict[str, Any] | None = None
     ```

9. `handle_tool_error`: Controls how `ToolException` is handled.

   It may re-raise the error, return its message, return a fixed string, or call a custom handler.

   * **Type:**
     ```python
     handle_tool_error: (
         bool
         | str
         | Callable[
             [ToolException],
             ToolExceptionHandlerOutput
         ]
         | None
     ) = False
     ```

10. `handle_validation_error`: Controls how Pydantic input-validation errors are handled.
    * **Type:**
      ```python
      handle_validation_error: (
          bool
          | str
          | Callable[
              [ValidationError | ValidationErrorV1],
              str
          ]
          | None
      ) = False
      ```

11. `response_format`: Specifies how the tool's return value is interpreted.

    `"content"` treats the return value as message content. `"content_and_artifact"` requires a two-item tuple containing content and an artifact.

    * **Type:**
      ```python
      response_format: Literal[
          "content",
          "content_and_artifact"
      ] = "content"
      ```

12. `extras`: Stores optional provider-specific tool configuration.
    * **Type:**
      ```python
      extras: dict[str, Any] | None = None
      ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `is_single_input`: Indicates whether the tool accepts only one non-variadic input argument.
   * **Type:**
     ```python
     is_single_input: bool
     ```

2. `args`: Returns the property definitions from the tool's input JSON schema.
   * **Type:**
     ```python
     args: dict[str, Any]
     ```

3. `tool_call_schema`: Returns the schema exposed to language models for generating tool calls.

   Runtime-injected arguments are excluded. The generated schema is cached and is rebuilt when `name`, `description`, or `args_schema` changes.

   * **Type:**
     ```python
     tool_call_schema: ArgsSchema
     ```

### Methods

1. `__init_subclass__`: Validates a tool subclass when the subclass is created.

   It raises `SchemaAnnotationError` for a common invalid `args_schema` annotation.

   * **Syntax:**
     ```python
     __init_subclass__(
         cls,
         **kwargs: Any # Arguments passed to the parent subclass initializer
     ) -> None
     ```

2. `__init__`: Initializes the tool and validates the supplied `args_schema`.

   A `TypeError` is raised when `args_schema` is neither a Pydantic model class nor a JSON Schema dictionary.

   * **Syntax:**
     ```python
     __init__(
         self,
         **kwargs: Any # Tool fields used during initialization
     ) -> None
     ```

3. `__setattr__`: Assigns an attribute and clears the cached tool-call schema when a schema-related field changes.
   * **Syntax:**
     ```python
     __setattr__(
         self,
         name: str, # Attribute name
         value: Any # New attribute value
     ) -> None
     ```

4. `model_copy`: Creates a copy of the tool.

   The cached tool-call schema is cleared when `name`, `description`, or `args_schema` is updated in the copy.

   * **Syntax:**
     ```python
     model_copy(
         self,
         *,
         update: Mapping[str, Any] | None = None, # Fields updated in the copy
         deep: bool = False # Whether to perform a deep copy
     ) -> Self
     ```

5. `__getstate__`: Returns the tool state used during pickling.

   The dynamically generated tool-call schema cache is removed because it cannot be pickled by reference.

   * **Syntax:**
     ```python
     __getstate__(
         self
     ) -> dict[Any, Any]
     ```

6. `get_input_schema`: Returns the schema used to validate the tool input.

   It returns `args_schema` when a Pydantic schema is supplied, delegates to the Runnable schema for JSON Schema dictionaries, or generates a schema from `_run`.

   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Runtime configuration
     ) -> TypeBaseModel
     ```

7. `invoke`: Executes the tool synchronously through the Runnable interface.

   A complete `ToolCall` input is separated into its arguments and tool-call identifier before execution.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: str | dict[str, Any] | ToolCall, # Tool input or complete tool call
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional execution arguments
     ) -> Any
     ```

8. `ainvoke`: Executes the tool asynchronously through the Runnable interface.
   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: str | dict[str, Any] | ToolCall, # Tool input or complete tool call
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional execution arguments
     ) -> Any
     ```

9. `_run`: Defines the synchronous operation performed by a concrete tool.

   Subclasses must implement this protected abstract method. A `run_manager` parameter may be added by subclasses to enable tracing.

   * **Syntax:**
     ```python
     @abstractmethod
     _run(
         self,
         *args: Any, # Positional tool arguments
         **kwargs: Any # Keyword tool arguments
     ) -> Any
     ```

10. `_arun`: Defines the asynchronous operation performed by the tool.

    The default implementation runs `_run` in an executor. Subclasses may override it to provide native asynchronous execution.

    * **Syntax:**
      ```python
      async _arun(
          self,
          *args: Any, # Positional tool arguments
          **kwargs: Any # Keyword tool arguments
      ) -> Any
      ```

11. `run`: Executes the tool synchronously with validation, callbacks, tracing, configuration, and error handling.

    When a `tool_call_id` is supplied, normal output is converted into a `ToolMessage` unless the output already implements `ToolOutputMixin`.

    * **Syntax:**
      ```python
      run(
          self,
          tool_input: str | dict[str, Any], # Input passed to the tool
          verbose: bool | None = None, # Whether to log execution progress
          start_color: str | None = "green", # Callback color used at startup
          color: str | None = "green", # Callback color used at completion
          callbacks: Callbacks = None, # Execution callbacks
          *,
          tags: list[str] | None = None, # Tags associated with the run
          metadata: dict[str, Any] | None = None, # Metadata associated with the run
          run_name: str | None = None, # Optional name of the run
          run_id: uuid.UUID | None = None, # Optional run identifier
          config: RunnableConfig | None = None, # Runtime configuration
          tool_call_id: str | None = None, # Corresponding model tool-call ID
          **kwargs: Any # Additional callback arguments
      ) -> Any
      ```

12. `arun`: Executes the tool asynchronously with validation, callbacks, tracing, configuration, and error handling.

    The output follows the same formatting and error-handling rules as `run`.

    * **Syntax:**
      ```python
      async arun(
          self,
          tool_input: str | dict[str, Any], # Input passed to the tool
          verbose: bool | None = None, # Whether to log execution progress
          start_color: str | None = "green", # Callback color used at startup
          color: str | None = "green", # Callback color used at completion
          callbacks: Callbacks = None, # Execution callbacks
          *,
          tags: list[str] | None = None, # Tags associated with the run
          metadata: dict[str, Any] | None = None, # Metadata associated with the run
          run_name: str | None = None, # Optional name of the run
          run_id: uuid.UUID | None = None, # Optional run identifier
          config: RunnableConfig | None = None, # Runtime configuration
          tool_call_id: str | None = None, # Corresponding model tool-call ID
          **kwargs: Any # Additional callback arguments
      ) -> Any
      ```

# InjectedToolArg

`InjectedToolArg` is an annotation marker for tool arguments that are supplied at runtime instead of being generated by a language model.

Arguments marked with this class are excluded from the tool-call schema sent to the model.

# InjectedToolCallId

`InjectedToolCallId` marks a tool argument that receives the current tool-call identifier at runtime.

A tool using this annotation must be invoked with a complete `ToolCall` that includes an identifier.

## Bases

- `InjectedToolArg`

# BaseToolkit

`BaseToolkit` is the abstract base class for a collection of related tools that work together for a specific task or external system.

## Bases

- `BaseModel`
- `ABC`

### Methods

1. `get_tools`: Returns all tools contained in the toolkit.

   Subclasses must implement this abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     get_tools(
         self
     ) -> list[BaseTool]
     ```